# Advanced RAG Systems #2 — Agentic RAG: the loop that checks its own work

Naive RAG fails silently: it retrieves once, answers, and nothing ever checks the evidence. This notebook runs the three working demos behind episode #2, on the same Nimbus Cloud question as episode #1:

**The question:** *What is the SLA (Service Level Agreement) refund percentage for enterprise downtime exceeding 4 hours?*

1. **The grader** — a rule-based judge that says RETRY or PASS, with a reason.
2. **The generic grader** — the same idea, but it derives what to check from the question itself, so it works on questions it was never designed for.
3. **Routing** — three decisions *before* any retrieval: retrieve or answer directly, which source, cheap path or full Layer 1.

Run the cells top to bottom (Runtime → Run all). The first run downloads the models (~90 MB each), then they are cached.

In [1]:
# One-time setup: install the two libraries this demo needs.
# sentence-transformers = embedding + reranking models; rank-bm25 = keyword search.
# The "%pip" magic installs into this notebook's own Python environment.
%pip install -q sentence-transformers rank-bm25

Note: you may need to restart the kernel to use updated packages.


## The corpus — the same 11 chunks as episode #1

Three plans with three different percentages (10% / 15% / 25%), an FAQ that talks *about* refunds but contains no number, and an overview page full of the right words with no answer. The traps are the point: a fixed pipeline answers from the FAQ and gets it wrong. The loop below catches that.

In [2]:
# The knowledge base: 11 chunks (passages) from a fictional Nimbus Cloud wiki.
# Each entry is (chunk_name, chunk_text). Traps are built in on purpose:
# three plans with three percentages (10% / 15% / 25%), an FAQ that talks about
# refunds but has NO number, and an overview page with the right words but no answer.
CORPUS = [
    ("sla_standard",
     "Nimbus Cloud SLA: Standard plan customers receive a 10% service credit when monthly "
     "downtime exceeds 1 hour. Credits are applied automatically to the next invoice. "
     "The Standard plan SLA covers compute and storage services only."),
    ("sla_enterprise",
     "Nimbus Cloud SLA: Enterprise tier customers receive a 25% service credit when monthly "
     "downtime exceeds 4 hours. The Enterprise SLA covers compute, storage, and networking. "
     "Claims must be filed within 30 days of the incident."),
    ("enterprise_support",
     "Enterprise support includes 24/7 phone support with a 15-minute response time for "
     "critical issues. Enterprise customers get a dedicated account manager and quarterly "
     "business reviews."),
    ("refund_policy",
     "Refund policy: monthly plans can be refunded within 7 days of purchase, annual plans "
     "within 30 days. Enterprise contracts are non-refundable except as stated in the SLA."),
    ("sla_pro",
     "Nimbus Cloud SLA: Pro plan customers receive a 15% service credit when monthly "
     "downtime exceeds 2 hours. Credits are applied automatically to the next invoice."),
    ("downtime_definition",
     "Downtime is measured as minutes when the service returns 5xx errors for over 60 "
     "consecutive seconds, excluding scheduled maintenance windows announced 72 hours in advance."),
    ("status_page",
     "Check status.nimbus.example for real-time availability. Historical uptime for the last "
     "90 days is published there with per-region breakdowns."),
    ("credits_process",
     "Service credits are applied to the next invoice automatically. Credits cannot be "
     "converted to cash refunds under any plan."),
    ("sla_overview",
     "Nimbus Cloud's SLA program reflects our commitment to reliability. Every plan includes "
     "service level objectives for uptime, measured monthly. Enterprise customers enjoy "
     "enhanced SLA terms with higher service credits for extended outages. See the "
     "plan-specific pages for exact percentages."),
    ("outage_postmortem",
     "Postmortem: the March 3 enterprise outage lasted 4.5 hours and SLA credits were issued "
     "per policy. Root cause was a load balancer misconfiguration. No action needed."),
    ("sla_faq",
     "SLA FAQ: what is an SLA refund? Service credits compensate customers for downtime. "
     "How do I claim? File a claim within 30 days. Which plans are covered? All paid plans "
     "include an SLA."),
]

# The question we will ask in every demo below.
QUERY = "What is the SLA refund percentage for enterprise downtime exceeding 4 hours?"
# The rewritten question the agent produces after the grader says RETRY (section 1).
REWRITTEN = "enterprise SLA service credit percentage downtime exceeding 4 hours"

def rrf(rank_lists, k=60):
    # Reciprocal Rank Fusion: merge several ranked lists into one.
    # A chunk earns 1 / (k + its position) in each list; ranks, not scores,
    # because BM25 scores and cosine scores live on different scales.
    scores = {}
    for ranks in rank_lists:
        for rank, idx in enumerate(ranks):
            scores[idx] = scores.get(idx, 0) + 1.0 / (k + rank + 1)
    return sorted(scores, key=scores.get, reverse=True)

## Section 1 — The grader: "can I answer from this, or not?"

The reranker ranks *relevance* — which chunk fits the question best. The grader asks a harder question: is the top chunk *sufficient* to answer? It checks three things:

- **Completeness:** does the chunk contain a percentage figure?
- **Scope:** does it mention the Enterprise plan?
- **Condition:** does it mention the 4-hour threshold?

A good grader returns more than pass/fail — a structured verdict (`decision / missing / reason`). The `missing` field is what makes rewriting possible: it tells the next step exactly what to fix.

In [3]:
# The embedding model turns text into vectors (lists of numbers).
# We load it once here and reuse it in the sections below.
from sentence_transformers import SentenceTransformer
import numpy as np
import re

emb = SentenceTransformer("all-MiniLM-L6-v2")  # small, free, runs on CPU
texts = [t for _, t in CORPUS]
E = emb.encode(texts, normalize_embeddings=True)  # one 384-number vector per chunk

def retrieve(q):
    # Embed the question, score every chunk in one matrix multiply, return the best chunk.
    v = emb.encode([q], normalize_embeddings=True)[0]
    return CORPUS[int(np.argmax(E @ v))]

def grade(question, chunk_id, chunk_text):
    # The grader: three deterministic checks, no LLM call. Returns a verdict dict.
    missing = []
    if not re.search(r"\d+\s*%", chunk_text):       # no percentage figure?
        missing.append("a percentage figure")
    if "enterprise" not in chunk_text.lower():      # wrong plan / no plan?
        missing.append("the Enterprise plan")
    if not re.search(r"4\s*hour", chunk_text.lower()):  # wrong condition?
        missing.append("the 4-hour condition")
    if missing:
        return {"decision": "RETRY", "missing": missing,
                "reason": f"top chunk [{chunk_id}] does not contain: {', '.join(missing)}"}
    return {"decision": "PASS", "missing": [],
            "reason": f"top chunk [{chunk_id}] has the number, the plan, and the condition"}

print(f"Question: {QUERY}")

# ---- Attempt 1: retrieve with the original question ----
cid, txt = retrieve(QUERY)
print(f"\n--- Attempt 1: retrieve with original question ---")
print(f"top-1: [{cid}]")
print(f'  "{txt[:110]}..."')
verdict = grade(QUERY, cid, txt)
print(f"GRADER: {verdict['decision']} — {verdict['reason']}")
print(f"  missing: {verdict['missing']}")

# ---- Attempt 2: rewrite the question, retrieve again ----
print(f'\n--- Rewrite: "{REWRITTEN}" ---')
cid, txt = retrieve(REWRITTEN)
print(f"top-1: [{cid}]")
print(f'  "{txt[:110]}..."')
verdict = grade(QUERY, cid, txt)
print(f"GRADER: {verdict['decision']} — {verdict['reason']}")

if verdict["decision"] == "PASS":
    pct = re.search(r"(\d+\s*%)", txt).group(1)
    print(f"\nANSWER: {pct} service credit (Enterprise SLA).")

/home/hatch/workspace/ep9x-venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:  73%|███████▎  | 75/103 [00:00<00:00, 679.98it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 904.45it/s]

Question: What is the SLA refund percentage for enterprise downtime exceeding 4 hours?

--- Attempt 1: retrieve with original question ---
top-1: [sla_faq]
  "SLA FAQ: what is an SLA refund? Service credits compensate customers for downtime. How do I claim? File a clai..."
GRADER: RETRY — top chunk [sla_faq] does not contain: a percentage figure, the Enterprise plan, the 4-hour condition
  missing: ['a percentage figure', 'the Enterprise plan', 'the 4-hour condition']

--- Rewrite: "enterprise SLA service credit percentage downtime exceeding 4 hours" ---
top-1: [sla_enterprise]
  "Nimbus Cloud SLA: Enterprise tier customers receive a 25% service credit when monthly downtime exceeds 4 hours..."
GRADER: PASS — top chunk [sla_enterprise] has the number, the plan, and the condition

ANSWER: 25% service credit (Enterprise SLA).


## Section 2 — The generic grader: no hardcoded answers

The grader above is hardcoded to the SLA question — it checks for "enterprise" and "4 hours" by name. What if the question is one we never designed for? This grader derives what to check **from the question itself**:

1. **Quantity-seeking?** If the question asks for a number (%, how much, how long, window, rate...), the chunk must contain a digit.
2. **Key-term coverage:** most content words of the question (stopwords removed, light stemming so "exceeding" matches "exceeds") must appear in the chunk.

No SLA, no "enterprise", no "4 hours" anywhere in the code. Two questions below: the SLA one, and a refund-window question against a small returns corpus.

In [4]:
# A second, smaller corpus: returns policy. Same idea, different documents.
REFUND_CORPUS = [
    ("returns_domestic",
     "Domestic returns: monthly plans can be refunded within 7 days of purchase, "
     "annual plans within 30 days. Refunds go back to the original payment method."),
    ("returns_intl",
     "International returns: customers can return international orders within 60 days "
     "of delivery for a full refund. Return shipping is paid by the customer."),
    ("returns_process",
     "To start a return, open a ticket with your order number. Approved refunds are "
     "processed in 5 business days."),
]

# Words that carry no meaning for matching ("what", "is", "the", ...) — skipped.
STOP = set("what is the a an of for to in on does do did can could would should "
           "which who whom whose when where why how".split())
# Words that signal the question wants a number.
QUANTITY_WORDS = {"percentage", "%", "how much", "how many", "how long",
                  "window", "rate", "price", "cost", "fee", "limit", "long"}

def stem(w):
    # Light stemming: strip a trailing -ing / -es / -s so "exceeding" matches "exceeds".
    for suf in ("ing", "es", "s"):
        if w.endswith(suf) and len(w) > len(suf) + 2:
            return w[: -len(suf)]
    return w

def key_terms(question):
    # The question's content words: lowercase, split, drop stopwords and tiny words.
    words = re.findall(r"[a-z0-9%]+", question.lower())
    return [w for w in words if w not in STOP and len(w) > 2]

def grade_generic(question, chunk_id, chunk_text):
    # The generic grader: checks derived from the question, not from any one document.
    missing = []
    q = question.lower()
    text = chunk_text.lower()
    # 1. Quantity-seeking question -> the chunk must contain a number.
    if any(w in q for w in QUANTITY_WORDS):
        if not re.search(r"\d+", text):
            missing.append("a number")
    # 2. Most key terms of the question must appear in the chunk.
    terms = key_terms(question)
    absent = [t for t in terms if stem(t) not in text]
    if len(absent) > len(terms) // 2:
        missing.append(f"key terms missing: {', '.join(absent)}")
    if missing:
        return {"decision": "RETRY", "missing": missing,
                "reason": f"top chunk [{chunk_id}] lacks: {', '.join(missing)}"}
    return {"decision": "PASS", "missing": [],
            "reason": f"top chunk [{chunk_id}] covers the question's terms and numbers"}

def retrieve_from(corpus, q):
    # Same dense retrieval as section 1, but over whichever corpus we pass in.
    texts = [t for _, t in corpus]
    E2 = emb.encode(texts, normalize_embeddings=True)
    v = emb.encode([q], normalize_embeddings=True)[0]
    return corpus[int(np.argmax(E2 @ v))]

cases = [
    ("What is the SLA refund percentage for enterprise downtime exceeding 4 hours?", CORPUS),
    ("What is the refund window for international orders?", REFUND_CORPUS),
]
for question, corpus in cases:
    print(f"\nQuestion: {question}")
    cid, txt = retrieve_from(corpus, question)
    print(f'  top-1: [{cid}] "{txt[:90]}..."')
    verdict = grade_generic(question, cid, txt)
    print(f"  GRADER: {verdict['decision']} — {verdict['reason']}")


Question: What is the SLA refund percentage for enterprise downtime exceeding 4 hours?


  top-1: [sla_faq] "SLA FAQ: what is an SLA refund? Service credits compensate customers for downtime. How do ..."
  GRADER: RETRY — top chunk [sla_faq] lacks: key terms missing: percentage, enterprise, exceeding, hours

Question: What is the refund window for international orders?
  top-1: [returns_intl] "International returns: customers can return international orders within 60 days of deliver..."
  GRADER: PASS — top chunk [returns_intl] covers the question's terms and numbers


## Section 3 — Routing: decide *before* retrieving

Sections 1–2 fix a bad retrieval. Routing asks an earlier question: should we retrieve at all? Three decisions, in order:

1. **Retrieve, or answer from the model itself?** "What does SLA stand for?" is general knowledge — answer directly. Retrieving would add latency and cost, and a stale chunk could corrupt a good answer.
2. **Which source?** Outages → postmortems (what happened). "My current bill" → the database (a live number). Policy questions → the wiki.
3. **Cheap path or full Layer 1?** Money or percentages at stake → the full engine (hybrid + RRF + rerank). Low stakes → dense search only.

This is the rules design — the cheapest router, no LLM call. The six questions below run through the router, then execute the chosen path for real.

In [5]:
# Two more sources for the router to choose between.
POSTMORTEM_CORPUS = [
    ("pm_march3",
     "Postmortem: the March 3 enterprise outage lasted 4.5 hours. Root cause was a "
     "load balancer misconfiguration. SLA credits were issued per policy."),
    ("pm_june12",
     "Postmortem: the June 12 database failover took 40 minutes. Root cause was a "
     "misconfigured connection pool. No data loss."),
]
# A live number lives in a database, not in documents — the router must know that.
DATABASE = {"current_bill": "$1,240.00, due Dec 1", "plan": "Enterprise"}

# Words that mark a question as money/percentage at stake -> full Layer 1.
MONEY_WORDS = {"%", "percentage", "percent", "refund", "credit", "bill", "price", "cost"}

def route(question):
    # The router: three decisions, in order. No retrieval happens inside this function.
    q = question.lower()
    reasons = []
    # --- Decision 1: retrieve, or answer directly? ---
    # Pure definition questions are general knowledge: answer directly, ~$0.
    if re.search(r"what does \w+ stand for", q) or (
        re.match(r"what is an? \w+\?$", q.strip()) and len(q.split()) <= 5
    ):
        return {"action": "answer_directly", "source": None, "path": None}, \
               ["definition question — general knowledge, no document needed"]
    reasons.append("asks for a specific fact, not general knowledge")
    # --- Decision 2: which source? ---
    if any(w in q for w in ("outage", "root cause", "incident", "postmortem")):
        source = "postmortems"
        reasons.append("asks what happened — postmortems, not policy docs")
    elif any(w in q for w in ("my", "bill", "invoice", "current")):
        source = "database"
        reasons.append("asks for a live number — database, not documents")
    else:
        source = "wiki"
        reasons.append("asks about policy — the wiki")
    # --- Decision 3: cheap path or full Layer 1? ---
    # (Only applies to document sources — a database lookup has no retrieval path.)
    if source == "database":
        path = None
        reasons.append("database lookup — no retrieval path applies")
    elif any(w in q for w in MONEY_WORDS):
        path = "full Layer 1"
        reasons.append("money/percentage at stake — full engine earns its cost")
    else:
        path = "cheap (dense only)"
        reasons.append("low stakes — cheap path is enough")
    return {"action": "retrieve", "source": source, "path": path}, reasons

# --- Execution: run the path the router chose, for real ---
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder

print("loading reranker...")
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
corpora = {"wiki": CORPUS, "postmortems": POSTMORTEM_CORPUS}
E_map = {name: emb.encode([t for _, t in c], normalize_embeddings=True)
         for name, c in corpora.items()}

def dense_top1(corpus_name, q):
    # Cheap path: dense search only, top-1.
    v = emb.encode([q], normalize_embeddings=True)[0]
    i = int(np.argmax(E_map[corpus_name] @ v))
    return corpora[corpus_name][i]

def full_layer1(q):
    # Full Layer 1 (episode #1): BM25 + dense -> RRF -> cross-encoder rerank, top-1.
    texts = [t for _, t in CORPUS]
    tok = [t.lower().split() for t in texts]
    v = emb.encode([q], normalize_embeddings=True)[0]
    cos = E_map["wiki"] @ v
    b = BM25Okapi(tok).get_scores(q.lower().split())
    fused = rrf([list(np.argsort(-b))[:8], list(np.argsort(-cos))[:8]])[:8]
    rs = reranker.predict([(q, texts[i]) for i in fused])
    return CORPUS[fused[int(np.argmax(rs))]]

questions = [
    "What does SLA stand for?",
    "What is the SLA refund percentage for enterprise downtime exceeding 4 hours?",
    "What was the root cause of last week's outage?",
    "What is my current bill?",
    "What is an SLA?",
    "What is the Pro plan's service credit?",
]
for q in questions:
    decision, reasons = route(q)
    parts = [decision["action"]]
    if decision["action"] == "retrieve":
        parts.append(f"source={decision['source']}")
        if decision["path"]:
            parts.append(f"path={decision['path']}")
    print(f"\nQ: {q}")
    print(f"  ROUTE: {' | '.join(parts)}")
    print(f"  WHY: {'; '.join(reasons)}")
    if decision["action"] == "answer_directly":
        print("  RESULT: answered from the model itself — no retrieval, ~$0")
    elif decision["source"] == "database":
        print(f"  RESULT: database lookup -> current bill: {DATABASE['current_bill']}")
    elif decision["path"] == "cheap (dense only)":
        cid, txt = dense_top1(decision["source"], q)
        print(f'  RESULT: dense-only top-1 [{cid}]: "{txt[:80]}..."')
    else:
        cid, txt = full_layer1(q)
        print(f'  RESULT: full Layer 1 top-1 [{cid}]: "{txt[:80]}..."')

loading reranker...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 1111.86it/s]


Q: What does SLA stand for?
  ROUTE: answer_directly
  WHY: definition question — general knowledge, no document needed
  RESULT: answered from the model itself — no retrieval, ~$0

Q: What is the SLA refund percentage for enterprise downtime exceeding 4 hours?
  ROUTE: retrieve | source=wiki | path=full Layer 1
  WHY: asks for a specific fact, not general knowledge; asks about policy — the wiki; money/percentage at stake — full engine earns its cost


  RESULT: full Layer 1 top-1 [sla_enterprise]: "Nimbus Cloud SLA: Enterprise tier customers receive a 25% service credit when mo..."

Q: What was the root cause of last week's outage?
  ROUTE: retrieve | source=postmortems | path=cheap (dense only)
  WHY: asks for a specific fact, not general knowledge; asks what happened — postmortems, not policy docs; low stakes — cheap path is enough
  RESULT: dense-only top-1 [pm_march3]: "Postmortem: the March 3 enterprise outage lasted 4.5 hours. Root cause was a loa..."

Q: What is my current bill?
  ROUTE: retrieve | source=database
  WHY: asks for a specific fact, not general knowledge; asks for a live number — database, not documents; database lookup — no retrieval path applies
  RESULT: database lookup -> current bill: $1,240.00, due Dec 1

Q: What is an SLA?
  ROUTE: answer_directly
  WHY: definition question — general knowledge, no document needed
  RESULT: answered from the model itself — no retrieval, ~$0

Q: What is the Pro plan's servi

  RESULT: full Layer 1 top-1 [sla_pro]: "Nimbus Cloud SLA: Pro plan customers receive a 15% service credit when monthly d..."


## Takeaway

Same corpus, same question — the difference between naive RAG and agentic RAG is the loop around the retrieval:

- **The grader** turns silent failure into a loud, specific verdict ("RETRY — missing a percentage figure"), and its `missing` field tells the agent what to fix.
- **Rewriting** fixes the question instead of blaming the search — but only what the question already implied, and only after a FAIL.
- **Routing** skips retrieval entirely when the model already knows, picks the right source when it doesn't, and spends the full engine only where the stakes justify it.

A pipeline runs the steps you wrote in advance. An agent writes its own steps as it goes — and checks its work.

*Inside the LLM — Advanced RAG Systems #2*